![logo](https://github.com/HelmholtzAI-Consultants-Munich/XAI-Tutorials/blob/main/docs/source/_figures/Helmholtz-AI.png?raw=true)

# Transformer Models

In this Notebook we will briefly introduce the concept of transformers in machine learning and show you how to train a transformer model.

---

## Getting Started

### Setup Colab environment

If you installed the packages and requirements on your  machine, you can skip this section and start from the import section.
Otherwise, you can follow and execute the tutorial on your browser. To start working on the notebook, click on the following button. This will open this page in the Colab environment, and you will be able to execute the code on your own.

<a href="https://colab.research.google.com/github/HelmholtzAI-Consultants-Munich/XAI-Tutorials/blob/main/xai-for-transformer/1-Tutorial_Transformer_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Now that you opened the notebook in Google Colab, follow the next step:

1. Run this cell to connect your Google Drive to Colab and install packages
2. Allow this notebook to access your Google Drive files. Click on 'Yes', and select your account.
3. "Google Drive for desktop wants to access your Google Account". Click on 'Allow'.
   
At this point, a folder has been created in your Drive, and you can navigate it through the lefthand panel in Colab. You might also receive an email that informs you about the access on your Google Drive.

In [1]:
# Mount drive folder to dbe abale to download repo
# from google.colab import drive
# drive.mount('/content/drive')

# Switch to correct folder'
# %cd /content/drive/MyDrive

In [2]:
# Don't run this cell if you already cloned the repo 
# %rm -r XAI-Tutorials
# !git clone --branch main https://github.com/HelmholtzAI-Consultants-Munich/XAI-Tutorials.git

In [3]:
# Install al required dependencies and package versions
# %cd XAI-Tutorials
# !pip install -r requirements_xai-for-transformer.txt
# %cd xai-for-transformer

### Imports

In [ ]:
import math

import numpy as np
import torch
import torch.nn as nn

from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

# Use GPU (CUDA), Apple Silicon (MPS), or CPU - whichever is available
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

--------

## Build a Transformer Model

In the subsequent sections we will show you how to build a transformer architecture using PyTorch, we alternate a step-by-step implementation with a small video explanation of the specific component.

***Note: we provide all references [here](https://xai-tutorials.readthedocs.io/en/latest/_ml_basics/transformer.html#references).***

### Motivation

Transformers largely replaced earlier sequence models (such as recurrent neural networks): their attention mechanism processes all tokens in parallel and captures long-range relationships between them directly, which makes them both faster to train and more accurate. This architecture underpins today's large language models. The video below builds this intuition and visualizes the overall architecture.

*TODO: Add video 1*

### Input Processing

Before an input sequence can be processed by a transformer model, it undergoes several preprocessing steps: 

- **Tokenization:** is a fundamental step in Natural Language Processing (NLP). It involves splitting text into smaller units called tokens. Depending on the tokenization strategy, tokens can be words, subwords, characters or even an entire sentence.

  Example:  
  Text: "Natural Language Processing is fascinating."  
  Tokens: ["Natural", "Language", "Processing", "is", "fascinating"]  
  *TODO: add line of code* 

- **Input Embeddings:** After tokenization, each token is converted into a high-dimensional vector representation called embedding. These embeddings capture semantic relationships between tokens and serve as numerical input to the transformer.

    *TODO: add line of code + add video 2*


- **Positional Encodings:** Unlike recurrent or convolutional networks, transformers do not inherently capture the order of tokens in a sequence. To provide positional information, positional encodings are added to the input embeddings. These encodings have the same dimensionality as the embeddings and are typically generated using sine and cosine functions of varying frequencies. This approach assigns a unique representation to each position, enabling the model to distinguish between tokens based on their order within the sequence.

    *TODO: add video 3*

In [ ]:
# Define the positional encoding layer
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()
        
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]
    

### Multi Head Attention

The Multi-Head Attention mechanism enables the Transformer to model relationships between different positions in a sequence. Instead of relying on a single attention operation, it uses multiple attention heads, each of which can focus on different aspects of the input and learn distinct patterns or dependencies.

Within each attention head, Scaled Dot-Product Attention is computed. The outputs of all attention heads are then concatenated and passed through a linear transformation to produce the final output representation.

Before diving into the implementation, it is helpful to first develop an intuitive understanding of the attention mechanism. The following video provides a clear explanation of how attention works and why it is a key component of Transformer models:

Attention Mechanism Explained: *TODO: Insert video link*

For a deeper understanding of the attention mechanisms used in the Transformer decoder, including Masked Self-Attention and Cross-Attention, you can also watch: (TODO: I would rather mention this after with the decoder part)

Attention in the Decoder (Masked and Cross-Attention): *TODO: Insert video link*

After reviewing these concepts, we will examine the code implementation of Multi-Head Attention in more detail.

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        # Ensure that the model dimension (d_model) is divisible by the number of heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        # Initialize dimensions
        self.d_model = d_model # Model's dimension
        self.num_heads = num_heads # Number of attention heads
        self.d_k = d_model // num_heads # Dimension of each head's key, query, and value
        
        # Linear layers for transforming inputs
        self.W_q = nn.Linear(d_model, d_model) # Query transformation
        self.W_k = nn.Linear(d_model, d_model) # Key transformation
        self.W_v = nn.Linear(d_model, d_model) # Value transformation
        self.W_o = nn.Linear(d_model, d_model) # Output transformation
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        # Calculate attention scores
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # Apply mask if provided (useful for preventing attention to certain parts like padding)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        
        # Softmax is applied to obtain attention probabilities
        attn_probs = torch.softmax(attn_scores, dim=-1)
        
        # Multiply by values to obtain the final output
        output = torch.matmul(attn_probs, V)
        return output
        
    def split_heads(self, x):
        # Reshape the input to have num_heads for multi-head attention
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        
    def combine_heads(self, x):
        # Combine the multiple heads back to original shape
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)
        
    def forward(self, Q, K, V, mask=None):
        # Apply linear transformations and split heads
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))
        
        # Perform scaled dot-product attention
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Combine heads and apply output transformation
        output = self.W_o(self.combine_heads(attn_output))
        return output

### Putting It All Together

The original Transformer has two components, an **encoder** and a **decoder**. Modern models often use only one, depending on the task. Since this course focuses on text *classification*, we will use the **encoder** only.

*TODO: add video*

#### Encoder

The encoder turns the input sequence into a contextualized representation that captures the relationships between tokens. It is a stack of identical layers, each containing:

* **Multi-Head Self-Attention**, which lets each token attend to all other tokens in the sequence.
* **Feed-Forward Network**, which refines each token's representation independently.
* **Residual Connections and Layer Normalization**, which stabilize training and help information flow.

The final layer outputs a sequence of vectors that encodes the meaning and context of the input, ready for a downstream task such as classification.

In [5]:
# The position-wise fully connected feed-forward network
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

# Define the encoder layer
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask):
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

#### Decoder

The **decoder** generates an output sequence one token at a time, using masked self-attention (so a token cannot attend to future positions) and cross-attention over the encoder's output. It is mainly used for sequence-generation tasks such as machine translation.

Since this course focuses on understanding and classifying text, **we do not use the decoder here.** Its full implementation, together with a machine-translation fine-tuning example, is covered in the companion notebook:

*TODO: add link to translation notebook (Old-1-Tutorial_Transformer_Model.ipynb).*

### Assemble the final Transformer Classifier

For the final Transformer classifier, we combine the positional encodings, multi-head self-attention mechanisms, and stacked encoder layers introduced above into a complete encoder-only architecture suitable for text classification.

- **Mask generation:** An attention mask is created for the input sequence to prevent the attention mechanism from focusing on padding tokens. This ensures that only meaningful tokens contribute to the learned representations.

- **Embedding and positional encoding:** The input tokens are mapped to dense vector representations through an embedding layer (`embedding`). A `PositionalEncoding` layer is then applied to inject positional information, enabling the model to capture the order of tokens within the sequence.

- **Encoder layers:** The model consists of multiple encoder layers, defined by `num_layers`. Each encoder layer contains multi-head self-attention, feed-forward networks, residual connections, and layer normalization. The input embeddings are progressively transformed into contextual representations that capture relationships between all tokens in the sequence.

- **Sequence representation and classification head:** After the encoder stack, the contextual representation of the input sequence is aggregated into a sequence-level representation. This representation is then passed through a fully connected classification head to predict the output class (e.g., positive or negative sentiment).

> **Note:** Our simplified implementation uses the representation of the first token as a sequence-level representation. In BERT and DistilBERT, a special classification token (`[CLS]`) is prepended to every input sequence, and its final contextual representation is used by the classification head.

In [9]:
# Define the transformer classifier model
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers,
                 d_ff, max_seq_length, num_classes, dropout):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)

        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        self.classifier = nn.Linear(d_model, num_classes)
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, input_ids):
        return (input_ids != 0).unsqueeze(1).unsqueeze(2)

    def forward(self, input_ids):
        mask = self.generate_mask(input_ids)

        x = self.embedding(input_ids)
        x = self.positional_encoding(x)
        x = self.dropout(x)

        for encoder_layer in self.encoder_layers:
            x = encoder_layer(x, mask)

        cls_representation = x[:, 0, :]
        logits = self.classifier(cls_representation)

        return logits

### Parametrize the Transformer Classifier

To instantiate the Transformer classifier, we define the main architectural parameters that control the size and complexity of the model:

- `vocab_size`: The size of the vocabulary, i.e., the total number of unique tokens (words or subwords) that the model can process.
- `d_model`: The dimensionality of the token embeddings and hidden representations throughout the model.
- `num_heads`: The number of attention heads used in each multi-head self-attention layer.
- `num_layers`: The number of stacked encoder layers.
- `d_ff`: The dimensionality of the hidden layer in the feed-forward network contained in each encoder layer.
- `max_seq_length`: The maximum input sequence length supported by the positional encoding.
- `num_classes`: The number of output classes for the classification task.
- `dropout`: The dropout probability used as a regularization technique to reduce overfitting.

The following parameters define a small Transformer classifier for demonstration purposes.

In [ ]:

vocab_size = 5000
d_model = 512
num_heads = 8
num_layers = 6
d_ff = 2048
max_seq_length = 100
num_classes = 2
dropout = 0.1


In [ ]:
transformer_classifier = TransformerClassifier(
    vocab_size,
    d_model,
    num_heads,
    num_layers,
    d_ff,
    max_seq_length,
    num_classes,
    dropout
).to(device)

## BERT

The previous sections introduced the original Transformer architecture and demonstrated how its main components can be assembled into an encoder-only Transformer classifier.

In practice, however, modern NLP applications rarely train Transformer models from scratch. Instead, they rely on **pretrained language models**, which are first trained on large text corpora and later adapted to specific downstream tasks through fine-tuning.

One of the most influential encoder-only Transformer models is **BERT (Bidirectional Encoder Representations from Transformers)**. Unlike the original encoder-decoder Transformer, BERT consists only of the encoder stack and is designed to learn contextual representations of text by attending to both the left and right context of every token simultaneously.

*TODO: add video*

In this course, we will use **DistilBERT**, a lightweight version of BERT that retains most of its performance while requiring fewer parameters and less computational resources. This makes it particularly suitable for educational purposes and explainability analyses.

*TODO: consider adding a table to show dofference between BERT and DistillBERT in readthedocs?*

### Loading a pretrained DistilBERT model

In [ ]:
checkpoint = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=6
    ).to(device)

## Extra Material: Fine-tune a Transformer Model

In the subsequent sections we will show you how to fine-tune a pre-trained Transformer model from [Hugging Face](https://huggingface.co/docs/transformers/index) on a text-classification task.

Our goal is to fine-tune the pre-trained **DistilBERT** model for an emotion-classification task, i.e. predicting the emotion expressed in a short piece of text.
We use `distilbert-base-uncased`, the lightweight version of BERT introduced above.
You can find the model card [here](https://huggingface.co/distilbert/distilbert-base-uncased).

In [ ]:
checkpoint = "distilbert-base-uncased"

The first step for fine-tuning a model is to load a dataset and prepare it for the classification task.
Here we use the [`dair-ai/emotion`](https://huggingface.co/datasets/dair-ai/emotion) dataset, which contains English tweets labeled with one of six emotions: *sadness, joy, love, anger, fear,* and *surprise*.

In [ ]:
emotion = load_dataset("dair-ai/emotion", "split")

labels = emotion["train"].features["label"].names
num_labels = len(labels)
id2label = {i: label for i, label in enumerate(labels)}
label2id = {label: i for i, label in enumerate(labels)}

labels

Next, we initialize a tokenizer based on the specified model checkpoint, which converts text into token IDs that the model can process.
We then define a pre-processing function that tokenizes each example's text (truncating long sequences) and apply it to the whole dataset.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_emotion = emotion.map(preprocess_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

For setting up the training we define the training arguments using the `TrainingArguments` class from the Hugging Face Transformers library.
The training arguments include settings such as the learning rate, batch size, number of epochs, evaluation strategy, and weight decay.

In [ ]:
training_args = TrainingArguments(
    output_dir="distilbert-emotion",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    num_train_epochs=3,
    save_total_limit=2,
    push_to_hub=False,
)

In addition to the training arguments, we define the evaluation metrics for our model.
For a classification task we report **accuracy** and the **weighted F1-score**, which balances precision and recall across the (imbalanced) emotion classes.

In [ ]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted"),
    }

Now we are ready to fine-tune DistilBERT for our emotion-classification task.
We initialize the model from the pre-trained checkpoint, adding a classification head sized to the six emotion labels.
The data collator dynamically pads the input sequences to the same length within each batch.

After initializing the `Trainer`, we can start the training/fine-tuning of the model.

*Note: for demonstration purposes we select only a small subsample of the train and validation data. The fully fine-tuned weights are provided below.*

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
).to(device)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_emotion["train"].select(range(200)),    # small subsample for demonstration
    eval_dataset=tokenized_emotion["validation"].select(range(50)),  # small subsample for demonstration
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

### Load the fully fine-tuned model

The demonstration above only fine-tunes on a small subsample, so its predictions are not yet reliable.
Fully fine-tuning DistilBERT on the complete emotion dataset takes ~20 minutes, so we provide the resulting weights as a downloadable asset.
The cell below loads them - reusing a local copy in `../data/` if one is already present, otherwise downloading it - and the rest of the tutorials use this fine-tuned model.

*The training script and metrics for these weights are in [`distilbert-emotion-training/`](./distilbert-emotion-training).*

In [ ]:
import os
import zipfile
import requests

# Fine-tuned DistilBERT emotion weights, hosted as a GitHub Release asset.
# NOTE: the download link becomes active once the weights Release is published.
url = "https://github.com/HelmholtzAI-Consultants-Munich/XAI-Tutorials/releases/download/distilbert-emotion-weights/distilbert_emotion_weights.zip"

weights_path = "../data/distilbert-emotion"
zip_path = "../data/distilbert_emotion_weights.zip"
os.makedirs("../data", exist_ok=True)

if not os.path.exists(weights_path):
    # Reuse a local zip if it is already there (e.g. shared for testing),
    # otherwise download it from the Release.
    if not os.path.exists(zip_path):
        with open(zip_path, "wb") as f:
            f.write(requests.get(url).content)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall("../data/")

model = AutoModelForSequenceClassification.from_pretrained(weights_path).to(device)
tokenizer = AutoTokenizer.from_pretrained(weights_path)